In [0]:
from pyspark.sql.functions import *

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
new_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/dbacademy/get_started_de/myfiles/orders_day2.csv")
)

In [0]:
new_orders.show()

In [0]:
customers_df = spark.table("bronze_customers")

products_df = spark.table("bronze_products")

In [0]:
customers_df.show()

products_df.show()

In [0]:
from pyspark.sql.functions import *

In [0]:
silver_orders_new = (
    new_orders
    .join(customers_df, "customer_id", "left")
    .join(products_df, "product_id", "left")
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )
    .withColumn(
        "order_date",
        to_date("order_date")
    )
)

In [0]:
silver_orders_new.show()

In [0]:
silver_orders = spark.table("silver_orders")

In [0]:
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders_delta")

In [0]:
spark.table("silver_orders_delta").show()

In [0]:
spark.sql("""
select count(*) as cnt
from silver_orders_delta
""").show()

In [0]:
from delta.tables import DeltaTable

In [0]:
delta_table = DeltaTable.forName(
    spark,
    "silver_orders_delta"
)

In [0]:
(
    delta_table.alias("target")
    .merge(
        silver_orders_new.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.table(
    "silver_orders_delta"
).orderBy("order_id").show()

In [0]:
spark.sql("""
select count(*) as cnt
from silver_orders_delta
""").show()